# 08 · Congestion and geographic visualisation

**Objective.** Look at the forecasts on the map: where was traffic observed to be slow, where did the models predict slow traffic, and where were they wrong?

> **Important.** Everything here is a **retrospective analysis of historical 2012 sensor data and model predictions**. It is *not* live traffic, and a low
> speed at a sensor does **not** by itself identify a particular accident or cause.

**Conventions used in every map.**
* One fixed speed colour scale for all maps, **0-70 mph, dark navy = slow / congested, pale = free flow**, so maps from different times, horizons and models are directly comparable.
* **Missing readings are grey crosses**, never coloured as a speed (so "no data" cannot be mistaken for "slow").
* Error maps show **predicted − actual** on a diverging scale: red = speed over-predicted, blue = speed under-predicted, grey cross = no ground truth.
* Sensor positions are real coordinates from `graph_sensor_locations.csv`.

**Methodology.**
1. Pick illustrative cases by a fixed rule (not by model performance) and draw observed vs predicted maps.
2. Rank the predicted-most-congested locations and compare with what was observed.
3. Quantify congestion flagging over the whole test period with a threshold fixed in advance.
4. Map where the errors (and the graph model's gain over XGBoost) are largest.
5. Provide an interactive map with a time slider.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

# Locate the project root (the folder that contains src/config.py) wherever Jupyter was started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))

from src import config
config.ensure_dirs()
print(f"project root : .../{ROOT.name}  (absolute paths are not printed)")
print(f"run mode     : {config.RUN_MODE}" + ("   <-- SMOKE TEST: outputs are NOT final results" if config.IS_SMOKE else ""))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import data_loader as dl, graph_utils as gu, preprocessing as pp, evaluation as ev, visualization as viz

viz.apply_style()
dl.require_raw_files()
ctx = pp.load_context()
graph = gu.load_graph_artifacts(expected_sensor_ids=ctx.sensor_ids)
loc = graph.locations
N, HZ = len(ctx.sensor_ids), config.HORIZONS

WANT = ["persistence", "xgboost", "gat"]
preds, y, ymask, origins, horizons, missing = ev.load_aligned(WANT, "test", ctx.speed, ctx.mask)
MODELS = [m for m in WANT if m in preds]
print("models available for the maps:", MODELS, "| missing:", missing or "none")
assert {"xgboost", "gat"} <= set(MODELS), "run notebooks 05 and 06 first"
if config.IS_SMOKE:
    print("*** SMOKE RUN: predictions come from a tiny debug configuration ***")

## 1 · Cases and helper functions

Two illustrative cases are fixed **in advance** (not selected by looking at where a model does well): a weekday evening rush hour and a Sunday late morning,
both inside the test period, forecast 30 minutes ahead. `case(origin_timestamp, horizon_steps)` returns everything needed for any other timestamp or
horizon of your choice (the nearest available forecast origin is used).

In [ ]:
H_DEFAULT = 6                                                      # 30 minutes ahead
CASES = {"weekday evening peak": "2012-06-12 17:00", "Sunday late morning": "2012-06-10 11:00"}

def case(origin_timestamp, horizon_steps=H_DEFAULT):
    """Arrays for one forecast: observed speed at the origin, the actual speed at origin + h, and every model's prediction."""
    ts = pd.Timestamp(origin_timestamp)
    k = int(np.argmin(np.abs((ctx.timestamps[origins] - ts).total_seconds())))          # nearest available origin
    o, j = int(origins[k]), horizons.index(horizon_steps)
    out = {"origin_time": ctx.timestamps[o], "target_time": ctx.timestamps[o + horizon_steps], "h": horizon_steps,
           "observed_now": ctx.speed[o], "actual": ctx.speed[o + horizon_steps], "actual_observed": ctx.mask[o + horizon_steps],
           "pred": {m: preds[m][k, j] for m in MODELS}, "k": k}
    return out

c = case(CASES["weekday evening peak"])
print(f"origin {c['origin_time']} -> forecast for {c['target_time']} ({c['h'] * 5} min ahead)")
print(f"sensors with an observed value at the target time: {int(c['actual_observed'].sum())} of {N}")

## 2 · Observed versus predicted maps

In [ ]:
def draw_case(c, title_prefix=""):
    fig, axes = plt.subplots(2, 3, figsize=(19, 11.5))
    now_ok = np.isfinite(c["observed_now"])
    viz.plot_speed_map(axes[0, 0], loc, c["observed_now"], f"Observed at forecast origin {c['origin_time']:%a %H:%M}", observed=now_ok)
    viz.plot_speed_map(axes[0, 1], loc, c["actual"], f"Observed {c['h'] * 5} min later ({c['target_time']:%a %H:%M})", observed=c["actual_observed"])
    viz.plot_speed_map(axes[0, 2], loc, c["pred"]["gat"], f"ST-GAT prediction for {c['target_time']:%H:%M}")
    viz.plot_speed_map(axes[1, 0], loc, c["pred"]["xgboost"], f"XGBoost prediction for {c['target_time']:%H:%M}")
    err = lambda m: np.where(c["actual_observed"], c["pred"][m] - c["actual"], np.nan)
    viz.plot_error_map(axes[1, 1], loc, err("gat"), "ST-GAT error (predicted - actual)")
    viz.plot_error_map(axes[1, 2], loc, err("xgboost"), "XGBoost error (predicted - actual)")
    fig.suptitle(f"{title_prefix} - historical data and model predictions (not live traffic)", fontsize=13, y=0.995)
    fig.tight_layout()
    return fig

name = "weekday evening peak"
c = case(CASES[name]); fig = draw_case(c, name.capitalize())
viz.save_figure(fig, "08_map_weekday_evening_peak"); plt.show()
for m in ("gat", "xgboost"):
    e = np.abs(c["pred"][m] - c["actual"])[c["actual_observed"]]
    print(f"{viz.style(m)['label']:10s}: mean |error| over the {int(c['actual_observed'].sum())} observed sensors = {e.mean():.2f} mph")

In [ ]:
name = "Sunday late morning"
c2 = case(CASES[name]); fig = draw_case(c2, name.capitalize())
viz.save_figure(fig, "08_map_sunday_late_morning"); plt.show()
for m in ("gat", "xgboost"):
    e = np.abs(c2["pred"][m] - c2["actual"])[c2["actual_observed"]]
    print(f"{viz.style(m)['label']:10s}: mean |error| over the {int(c2['actual_observed'].sum())} observed sensors = {e.mean():.2f} mph")

**Reading the maps.** The first two panels show how the observed pattern changes in 30 minutes; the third and fourth show what the models predicted for that
later time on the same colour scale; the bottom-right panels show where each model over- or under-estimated speed. These are two cases, not statistics: the
whole-period quantitative comparison is in notebook 07 and section 4 below.

## 3 · Which locations are predicted to be most congested?

In [ ]:
def rank_table(c, model, top=10):
    pred = c["pred"][model]
    df = pd.DataFrame({"sensor_id": ctx.sensor_ids, "latitude": loc["latitude"].to_numpy(), "longitude": loc["longitude"].to_numpy(),
                       "predicted_mph": pred, "observed_mph": c["actual"], "observed_available": c["actual_observed"]})
    return df.sort_values("predicted_mph").head(top).reset_index(drop=True)

for m in ("gat", "xgboost"):
    print(f"--- {viz.style(m)['label']}: 10 lowest predicted speeds for {c['target_time']} (30 min ahead) ---")
    display(rank_table(c, m).round(2))

def overlap_at_k(c, model, k=10):
    obs = np.where(c["actual_observed"], c["actual"], np.inf)
    truly = set(np.argsort(obs)[:k][np.isfinite(np.sort(obs)[:k])])
    return len(truly & set(np.argsort(c["pred"][model])[:k])), len(truly)
for m in ("persistence", "xgboost", "gat"):
    hit, tot = overlap_at_k(c, m)
    print(f"{viz.style(m)['label']:12s}: {hit} of the {tot} truly slowest observed sensors are among its 10 predicted-slowest")

## 4 · Congestion flagging over the whole test period

Retrospective check of how well *"predicted speed below a threshold"* identifies sensors that were actually slow. The threshold (**35 mph**, a commonly used
freeway congestion convention) was fixed before looking at any result and is the same for all models. This evaluates speed-threshold flags only; it does
not identify incidents.

In [ ]:
THRESHOLD = 35.0
all_pred = {m: ev.clip_predictions(preds[m]) for m in MODELS}
valid = ev.common_valid_mask(all_pred, ymask)
rows = []
for m in MODELS:
    for j, h in enumerate(HZ):
        v = valid[:, j]; actual_slow = (y[:, j] < THRESHOLD) & v; flagged = (all_pred[m][:, j] < THRESHOLD) & v
        tp = int((actual_slow & flagged).sum()); fp = int((~actual_slow & flagged & v).sum()); fn = int((actual_slow & ~flagged).sum())
        prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
        rows.append({"model": m, "horizon_min": h * 5, "precision": prec, "recall": rec, "F1": 2 * prec * rec / max(prec + rec, 1e-9),
                     "share of valid points actually < 35 mph": actual_slow.sum() / v.sum()})
flag = pd.DataFrame(rows)
flag.to_csv(config.METRICS_DIR / f"{config.tagged('congestion_flagging')}.csv", index=False)
display(flag.pivot(index="model", columns="horizon_min", values=["precision", "recall", "F1"]).reindex(MODELS).round(3))
print("base rate (share of valid points that are truly below the threshold):", flag.groupby("horizon_min")["share of valid points actually < 35 mph"].first().round(3).to_dict())

fig, ax = plt.subplots(figsize=(8, 4.2))
for m in MODELS:
    d = flag[flag.model == m]; st = viz.style(m)
    ax.plot(d["horizon_min"], d["F1"], color=st["color"], marker=st["marker"], ls=st["ls"], label=st["label"])
ax.set_xlabel("forecast horizon (minutes)"); ax.set_ylabel("F1 of the flag 'speed < 35 mph'"); ax.set_xticks([h * 5 for h in HZ])
ax.set_title("Congestion flagging quality by horizon (test period)"); ax.legend()
viz.save_figure(fig, "08_congestion_flag_f1"); plt.show()

In [ ]:
# Ranking quality: how many of the 10 truly slowest observed sensors does each model place among its 10 predicted-slowest?
K = 10; rows = []
for j, h in enumerate(HZ):
    obs_rank = np.where(ymask[:, j], y[:, j], np.inf)                                # [S, N]
    enough = (ymask[:, j].sum(1) >= 2 * K)
    truly = np.argpartition(obs_rank, K, axis=1)[:, :K]
    for m in MODELS:
        top_pred = np.argpartition(all_pred[m][:, j], K, axis=1)[:, :K]
        hits = np.array([len(set(a) & set(b)) for a, b in zip(truly[enough], top_pred[enough])])
        rows.append({"model": m, "horizon_min": h * 5, "mean overlap (of 10)": hits.mean(), "origins used": int(enough.sum())})
display(pd.DataFrame(rows).pivot(index="model", columns="horizon_min", values="mean overlap (of 10)").reindex(MODELS).round(2))

**Reading the tables.** Precision is the share of flagged sensors that really were below 35 mph, recall the share of truly slow sensors that were flagged.
The overlap table asks the ranking question directly. Both are retrospective, per-sensor speed statements; a low-speed sensor is not evidence of a specific incident.

## 5 · Where are the errors? (per-sensor maps, test period)

In [ ]:
j = HZ.index(H_DEFAULT)
per = {m: ev.per_sensor_metrics(all_pred[m], y, valid, HZ, ctx.sensor_ids) for m in ("xgboost", "gat")}
mae = {m: per[m][per[m].horizon_steps == H_DEFAULT].set_index("node")["MAE"].to_numpy() for m in per}
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
vmax = float(max(mae["xgboost"].max(), mae["gat"].max()))
for ax, m in zip(axes[:2], ("xgboost", "gat")):
    lon, lat = loc["longitude"].to_numpy(), loc["latitude"].to_numpy()
    sc = ax.scatter(lon, lat, c=mae[m], cmap="Blues", vmin=0, vmax=vmax, s=40, edgecolor=viz.INK_2, linewidth=0.4)
    ax.set_title(f"{viz.style(m)['label']}: per-sensor MAE, {H_DEFAULT * 5} min (test)"); ax.set_aspect(1 / np.cos(np.deg2rad(lat.mean())))
    ax.set_xlabel("Longitude (deg)"); ax.set_ylabel("Latitude (deg)"); plt.colorbar(sc, ax=ax, shrink=0.75, label="MAE (mph)")
diff = mae["xgboost"] - mae["gat"]                                       # positive = ST-GAT has the lower error at that sensor
lim = float(np.nanmax(np.abs(diff)))
sc = axes[2].scatter(lon, lat, c=diff, cmap=viz.ERROR_CMAP.reversed(), vmin=-lim, vmax=lim, s=40, edgecolor=viz.INK_2, linewidth=0.4)
axes[2].set_title("MAE(XGBoost) - MAE(ST-GAT) per sensor\n(positive = ST-GAT has the lower error)")
axes[2].set_aspect(1 / np.cos(np.deg2rad(lat.mean()))); axes[2].set_xlabel("Longitude (deg)"); axes[2].set_ylabel("Latitude (deg)")
plt.colorbar(sc, ax=axes[2], shrink=0.75, label="mph (positive = ST-GAT lower error)")
fig.tight_layout(); viz.save_figure(fig, "08_error_maps"); plt.show()
print(f"sensors where ST-GAT has the lower MAE at {H_DEFAULT * 5} min: {int((diff > 0).sum())} of {N}; median difference {np.median(diff):.3f} mph")

## 6 · Interactive map (time slider)

Three linked maps for the weekday of case 1: **observed** speed at the target time, the **XGBoost** prediction and the **ST-GAT** prediction, with a slider over forecast
origins (every 30 minutes) and the horizon fixed per figure. Grey markers are missing observations. The figure is also saved as a standalone HTML file
(`artifacts/figures/08_interactive_forecast_map_*.html`; base-map tiles need an internet connection when the HTML is opened).

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from matplotlib.colors import to_hex

colorscale = [[p, to_hex(viz.SPEED_CMAP(p))] for p in np.linspace(0, 1, 8)]

def interactive_map(horizon_steps, day="2012-06-12", every_minutes=30):
    d0 = pd.Timestamp(day)
    ks = [k for k in range(len(origins)) if d0 <= ctx.timestamps[origins[k]] < d0 + pd.Timedelta(days=1) and ctx.timestamps[origins[k]].minute % every_minutes == 0]
    if not ks:
        print("no forecast origins on that day (smoke mode keeps only a subset)"); return None
    j = horizons.index(horizon_steps)
    lon, lat = loc["longitude"].to_numpy(), loc["latitude"].to_numpy()
    def frame_traces(k):
        o = int(origins[k]); tgt = ctx.speed[o + horizon_steps]; ok = ctx.mask[o + horizon_steps]
        text = lambda v: [f"sensor {s}<br>{x:.1f} mph" for s, x in zip(ctx.sensor_ids, v)]
        mk = lambda c: dict(color=c, colorscale=colorscale, cmin=0, cmax=70, size=9, showscale=False)
        return [
            go.Scattermap(lon=lon[ok], lat=lat[ok], mode="markers", marker=mk(tgt[ok]), text=np.array(text(tgt))[ok], hoverinfo="text", name="observed"),
            go.Scattermap(lon=lon[~ok], lat=lat[~ok], mode="markers", marker=dict(color=viz.MISSING_COLOR, size=9), text=["missing observation"] * int((~ok).sum()), hoverinfo="text", name="missing"),
            go.Scattermap(lon=lon, lat=lat, mode="markers", marker=mk(ev.clip_predictions(preds["xgboost"][k, j])), text=text(ev.clip_predictions(preds["xgboost"][k, j])), hoverinfo="text", name="XGBoost"),
            go.Scattermap(lon=lon, lat=lat, mode="markers", marker=mk(ev.clip_predictions(preds["gat"][k, j])), text=text(ev.clip_predictions(preds["gat"][k, j])), hoverinfo="text", name="ST-GAT"),
        ]
    fig = make_subplots(rows=1, cols=3, specs=[[{"type": "map"}] * 3], subplot_titles=("Observed (historical)", "XGBoost forecast", "ST-GAT forecast"), horizontal_spacing=0.01)
    first = frame_traces(ks[0])
    for tr, col in zip(first, (1, 1, 2, 3)):
        fig.add_trace(tr, row=1, col=col)
    fig.frames = [go.Frame(data=frame_traces(k), traces=[0, 1, 2, 3], name=str(i)) for i, k in enumerate(ks)]
    centre = dict(lat=float(lat.mean()), lon=float(lon.mean()))
    fig.update_layout(map=dict(style="carto-positron", center=centre, zoom=9.2), map2=dict(style="carto-positron", center=centre, zoom=9.2), map3=dict(style="carto-positron", center=centre, zoom=9.2),
                      margin=dict(l=5, r=5, t=60, b=5), height=520, showlegend=False,
                      title=f"{horizon_steps * 5}-minute-ahead forecasts vs observed speed - historical data, not live traffic (colour: 0 mph dark -> 70 mph pale; grey = missing)",
                      sliders=[dict(active=0, pad=dict(t=10), currentvalue=dict(prefix="forecast origin: "),
                                    steps=[dict(method="animate", label=f"{ctx.timestamps[origins[k]]:%H:%M}", args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for i, k in enumerate(ks)])])
    return fig

for hs in (3, 6):
    figi = interactive_map(hs)
    if figi is not None:
        path = config.FIGURES_DIR / f"{config.tagged('08_interactive_forecast_map')}_h{hs}.html"
        figi.write_html(path, include_plotlyjs="cdn")
        print("saved", path.relative_to(config.PROJECT_ROOT))
        if hs == 6:
            figi.show()

**Reading the interactive map.** Drag the slider to move through the day: the left map shows what was observed at the target time, the other two what each model forecast for
that time from data available 15 or 30 minutes earlier. Hover over a marker for the sensor id and speed. As everywhere in this notebook the maps describe sensor speeds only
and say nothing about specific incidents.

## 7 · Summary

In [ ]:
print("figures written:", sorted(p.name for p in config.FIGURES_DIR.glob(f"{config.tagged('08_')}*")))
print("metrics written:", sorted(p.name for p in config.METRICS_DIR.glob(f"{config.tagged('congestion_flagging')}*")))
print("\nNext: 09_optional_sumo_simulation.ipynb (optional) - and review reports/results_summary.md")